# **Exploratory Data Analysis (EDA)**
**Decodelabs Internship | Week 1 | Task 3**

This notebook explores the cleaned Heart Disease dataset. I calculated statistics, examine distributions, check for outliers, and investigate relationships between features and the target variable.


## Importing libraries and loading data

In [1]:
import os
import pandas as pd
import numpy as np

NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
CLEAN_FILE   = os.path.join(PROJECT_ROOT, "data", "processed", "heart_cleveland_clean.csv")
TABLES_DIR   = os.path.join(PROJECT_ROOT, "reports", "tables")
os.makedirs(TABLES_DIR, exist_ok=True)

df = pd.read_csv(CLEAN_FILE)

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Dataset loaded: 297 rows × 14 columns


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1,1,145.0,233.0,1,2,150.0,0,2.3,3,0,6,0
1,67.0,1,4,160.0,286.0,0,2,108.0,1,1.5,2,3,3,1
2,67.0,1,4,120.0,229.0,0,2,129.0,1,2.6,2,2,7,1
3,37.0,1,3,130.0,250.0,0,0,187.0,0,3.5,3,0,3,0
4,41.0,0,2,130.0,204.0,0,2,172.0,0,1.4,1,0,3,0


## Defining feature groups

In [2]:
# I grouped the features by their type.
# This makes it easier to apply the right analysis to the right columns.

# Continuous (truly numeric, can take many values)
continuous_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

# Binary (only 0 or 1)
binary_features = ["sex", "fbs", "exang"]

# Categorical (a small number of integer codes)
categorical_features = ["cp", "restecg", "slope", "ca", "thal"]

target_col = "target"

print("Continuous features  :", continuous_features)
print("Binary features      :", binary_features)
print("Categorical features :", categorical_features)
print("Target               :", target_col)

Continuous features  : ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Binary features      : ['sex', 'fbs', 'exang']
Categorical features : ['cp', 'restecg', 'slope', 'ca', 'thal']
Target               : target


## Class balance (target distribution)

In [3]:
# I check how balanced the two classes are.
# A heavily imbalanced dataset (e.g. 95% class 0, 5% class 1) would require special handling in modelling. Here I check if that is the case.

counts = df[target_col].value_counts()
pcts   = df[target_col].value_counts(normalize=True) * 100

print("=== Target Variable Distribution ===")
print(f"  No heart disease (0): {counts[0]}  ({pcts[0]:.1f}%)")
print(f"  Heart disease    (1): {counts[1]}  ({pcts[1]:.1f}%)")

=== Target Variable Distribution ===
  No heart disease (0): 160  (53.9%)
  Heart disease    (1): 137  (46.1%)


The dataset is reasonably balanced, which means accuracy is a useful metric.

## Descriptive statistics for continuous features

In [4]:
print("=== Descriptive Statistics: Continuous Features ===")
df[continuous_features].describe().round(2)

=== Descriptive Statistics: Continuous Features ===


,age,trestbps,chol,thalach,oldpeak
count,297.00,297.00,297.00,297.00,297.00
mean,54.54,131.69,247.35,149.60,1.06
std,9.05,17.76,52.00,22.94,1.17
min,29.00,94.00,126.00,71.00,0.00
25%,48.00,120.00,211.00,133.00,0.00
50%,56.00,130.00,243.00,153.00,0.80
75%,61.00,140.00,276.00,166.00,1.60
max,77.00,200.00,564.00,202.00,6.20


In [5]:
# I compared these statistics split by the target variable.
# This tells me whether the average values differ between patients with and without heart disease.

print("=== Group Comparison: Continuous Features (mean ± std) ===")
group_stats = df.groupby(target_col)[continuous_features].agg(["mean", "std"]).round(2)
print(group_stats)

=== Group Comparison: Continuous Features (mean ± std) ===
          age       trestbps           chol        thalach        oldpeak  \
         mean   std     mean    std    mean    std    mean    std    mean   
target                                                                      
0       52.64  9.55   129.18  16.37  243.49  53.76  158.58  19.04    0.60   
1       56.76  7.90   134.64  18.90  251.85  49.68  139.11  22.71    1.59   

              
         std  
target        
0       0.79  
1       1.31  


Differences in means do not prove that a feature is a reliable predictor, but they suggest which features are worth examining more closely.

In [6]:
# I saved the group comparison table for the report.
group_mean = df.groupby(target_col)[continuous_features].mean().round(2)
group_mean.index = ["No Disease", "Disease"]
group_mean.to_csv(os.path.join(TABLES_DIR, "group_comparison_continuous.csv"))
print("Group comparison table saved.")
group_mean

Group comparison table saved.


,age,trestbps,chol,thalach,oldpeak
No Disease,52.64,129.18,243.49,158.58,0.60
Disease,56.76,134.64,251.85,139.11,1.59


## Outlier detection using IQR

In [7]:
# The IQR (Interquartile Range) method defines outliers as values that fall
# below Q1 - 1.5*IQR  or  above Q3 + 1.5*IQR.
# 
# I check this for every continuous feature. I will NOT automatically remove outliers, I will only note them and decide later whether they are errors or genuine extreme values.

print("=== Outlier Detection (IQR Method) ===")
for col in continuous_features:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    pct = n_outliers / len(df) * 100
    
    print(f"  {col:12s}: {n_outliers:3d} outliers ({pct:.1f}%) "
          f"| range [{lower_bound:.1f}, {upper_bound:.1f}]")

print()

=== Outlier Detection (IQR Method) ===
  age         :   0 outliers (0.0%) | range [28.5, 80.5]
  trestbps    :   9 outliers (3.0%) | range [90.0, 170.0]
  chol        :   5 outliers (1.7%) | range [113.5, 373.5]
  thalach     :   1 outliers (0.3%) | range [83.5, 215.5]
  oldpeak     :   5 outliers (1.7%) | range [-2.4, 4.0]



## Categorical feature distributions

In [8]:
# I look at the distribution of each categorical feature, split by whether the patient has heart disease or not.
# This tells me whether certain categories are more common in one group.

print("=== Categorical Feature Distributions by Target ===")
for col in categorical_features + binary_features:
    ct = pd.crosstab(df[col], df[target_col], margins=True)
    ct.columns = ["No Disease", "Disease", "Total"]
    print(f"\n-- {col} --")
    print(ct)

=== Categorical Feature Distributions by Target ===

-- cp --
     No Disease  Disease  Total
cp                             
1            16        7     23
2            40        9     49
3            65       18     83
4            39      103    142
All         160      137    297

-- restecg --
         No Disease  Disease  Total
restecg                            
0                92       55    147
1                 1        3      4
2                67       79    146
All             160      137    297

-- slope --
       No Disease  Disease  Total
slope                            
1             103       36    139
2              48       89    137
3               9       12     21
All           160      137    297

-- ca --
     No Disease  Disease  Total
ca                             
0           129       45    174
1            21       44     65
2             7       31     38
3             3       17     20
All         160      137    297

-- thal --
      No Disease  Di

## Correlation analysis

In [9]:
# The correlation matrix shows how strongly pairs of features are linearly related.
# Values close to +1 or -1 indicate strong relationships.
# Values close to 0 indicate weak or no linear relationship.

print("=== Correlation with Target Variable ===")
corr_with_target = df.corr()["target"].drop("target").sort_values(key=abs, ascending=False)
print(corr_with_target.round(3))

=== Correlation with Target Variable ===
thal        0.527
ca          0.463
oldpeak     0.424
thalach    -0.424
exang       0.421
cp          0.409
slope       0.333
sex         0.278
age         0.227
restecg     0.166
trestbps    0.153
chol        0.080
fbs         0.003
Name: target, dtype: float64


Positive values: feature tends to be higher in disease cases. Negative values: feature tends to be lower in disease cases.

In [10]:
# Full correlation matrix — I check for highly correlated feature PAIRS.
# If two features are very highly correlated (e.g. r > 0.9), they carry redundant information and I may want to keep only one.

corr_matrix = df.corr().round(3)
print("=== Full Correlation Matrix ===")
corr_matrix

=== Full Correlation Matrix ===


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
age,1.000,-0.092,0.110,0.290,0.203,0.132,0.150,-0.395,0.096,0.197,0.159,0.362,0.127,0.227
sex,-0.092,1.000,0.009,-0.066,-0.198,0.039,0.034,-0.060,0.144,0.107,0.033,0.092,0.384,0.278
cp,0.110,0.009,1.000,-0.037,0.072,-0.058,0.064,-0.339,0.378,0.203,0.151,0.236,0.268,0.409
trestbps,0.290,-0.066,-0.037,1.000,0.132,0.181,0.149,-0.049,0.067,0.191,0.121,0.098,0.138,0.153
chol,0.203,-0.198,0.072,0.132,1.000,0.013,0.165,-0.000,0.059,0.039,-0.009,0.116,0.011,0.080
fbs,0.132,0.039,-0.058,0.181,0.013,1.000,0.069,-0.008,-0.001,0.008,0.048,0.152,0.062,0.003
restecg,0.150,0.034,0.064,0.149,0.165,0.069,1.000,-0.072,0.082,0.114,0.135,0.129,0.019,0.166
thalach,-0.395,-0.060,-0.339,-0.049,-0.000,-0.008,-0.072,1.000,-0.384,-0.348,-0.389,-0.269,-0.275,-0.424
exang,0.096,0.144,0.378,0.067,0.059,-0.001,0.082,-0.384,1.000,0.289,0.251,0.148,0.327,0.421
oldpeak,0.197,0.107,0.203,0.191,0.039,0.008,0.114,-0.348,0.289,1.000,0.579,0.294,0.345,0.424


## Age distribution analysis

In [11]:
# Age is likely an important feature. I checked it carefully.

print("=== Age Distribution ===")
print(f"  Min age    : {df['age'].min()}")
print(f"  Max age    : {df['age'].max()}")
print(f"  Mean age   : {df['age'].mean():.1f}")
print(f"  Median age : {df['age'].median():.1f}")
print()

# Age by disease group
print("=== Age by Target Group ===")
print(df.groupby("target")["age"].describe().round(2))

=== Age Distribution ===
  Min age    : 29.0
  Max age    : 77.0
  Mean age   : 54.5
  Median age : 56.0

=== Age by Target Group ===
        count   mean   std   min    25%   50%   75%   max
target                                                   
0       160.0  52.64  9.55  29.0  44.75  52.0  59.0  76.0
1       137.0  56.76  7.90  35.0  53.00  58.0  62.0  77.0


## Sex and heart disease

In [12]:
# I examine whether sex is associated with disease presence.
# Clinical literature suggests men have higher rates of heart disease.

sex_counts = pd.crosstab(df["sex"], df["target"])
sex_counts.index = ["Female (0)", "Male (1)"]
sex_counts.columns = ["No Disease", "Disease"]
sex_counts["Disease Rate %"] = (sex_counts["Disease"] / 
                                 sex_counts.sum(axis=1) * 100).round(1)
print("=== Heart Disease by Sex ===")
print(sex_counts)

=== Heart Disease by Sex ===
            No Disease  Disease  Disease Rate %
Female (0)          71       25            26.0
Male (1)            89      112            55.7


## Key numerical findings

In [13]:
print("=== Key EDA Findings ===")
print()
print("1. Class balance:")
print(f"   {(df.target==0).sum()} no-disease vs {(df.target==1).sum()} disease cases. Reasonable balance.")
print()
print("2. Features most correlated with target (top 5):")
top5 = corr_with_target.abs().nlargest(5)
for feat, val in top5.items():
    direction = "+" if corr_with_target[feat] > 0 else "-"
    print(f"   {feat:12s}  r = {direction}{abs(val):.3f}")
print()
print("3. Age:")
print(f"   Disease group mean age  : {df[df.target==1]['age'].mean():.1f}")
print(f"   No-disease group mean age: {df[df.target==0]['age'].mean():.1f}")
print()
print("4. Maximum heart rate (thalach):")
print(f"   Disease group mean thalach  : {df[df.target==1]['thalach'].mean():.1f}")
print(f"   No-disease group mean thalach: {df[df.target==0]['thalach'].mean():.1f}")
print("   Lower max heart rate is associated with disease presence.")
print()
print("5. Sex distribution:")
male_disease_rate   = (df[df.sex==1].target==1).mean()*100
female_disease_rate = (df[df.sex==0].target==1).mean()*100
print(f"   Male disease rate  : {male_disease_rate:.1f}%")
print(f"   Female disease rate: {female_disease_rate:.1f}%")

=== Key EDA Findings ===

1. Class balance:
   160 no-disease vs 137 disease cases. Reasonable balance.

2. Features most correlated with target (top 5):
   thal          r = +0.527
   ca            r = +0.463
   oldpeak       r = +0.424
   thalach       r = -0.424
   exang         r = +0.421

3. Age:
   Disease group mean age  : 56.8
   No-disease group mean age: 52.6

4. Maximum heart rate (thalach):
   Disease group mean thalach  : 139.1
   No-disease group mean thalach: 158.6
   Lower max heart rate is associated with disease presence.

5. Sex distribution:
   Male disease rate  : 55.7%
   Female disease rate: 26.0%


These patterns are consistent with established clinical knowledge, which gives me confidence the dataset is well-formed.